In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

### The main goal of "Multi Query Translation" is to take the input question and to translate it in some way, as to improve retrieval

## Setup and Indexing (Common to all)

In [2]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
# Load blog

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

blog_docs = loader.load()

In [4]:
# Split 
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300, 
    chunk_overlap=50)

# Make splits
splits = text_splitter.split_documents(blog_docs)

In [5]:
# Index 

vectorstore = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings(),
    location=":memory:", 
    collection_name="my_documents"
)

retriever = vectorstore.as_retriever()

## 1) Multi-Query

![Multi-Query Image](Images/Multi-Query.png)

#### Multi-Query generates multiple alternative versions of a user question to improve document retrieval.

### Query Generation

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

In [7]:
# Multi Query: Different Perspectives 

#Prompt to generate 5 variations 

template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}""" 

prompt_perspectives = ChatPromptTemplate.from_template(template)

In [8]:
# Chain to generate the queries and split them into a Python list 

generate_queries = (
    prompt_perspectives 
    | ChatOpenAI(model="gpt-4o-mini", temperature=0) 
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

In [9]:
# This prints the LangChain Runnable (a chain object) itself

print(generate_queries)

first=ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='You are an AI language model assistant. Your task is to generate five \ndifferent versions of the given user question to retrieve relevant documents from a vector \ndatabase. By generating multiple perspectives on the user question, your goal is to help\nthe user overcome some of the limitations of the distance-based similarity search. \nProvide these alternative questions separated by newlines. Original question: {question}'), additional_kwargs={})]) middle=[ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True

In [10]:
# To see the actual multiple versions made by the LLM, we do this

question = "What is task decomposition for LLM agents?"
multi_queries = generate_queries.invoke({"question": question})

print(multi_queries)

['What does task decomposition mean in the context of LLM agents?  ', 'How do LLM agents utilize task decomposition in their processes?  ', 'Can you explain the concept of task decomposition as it relates to LLM agents?  ', 'In what ways is task decomposition applied by LLM agents?  ', 'What are the benefits of task decomposition for agents using large language models?  ']


### Retrieval Chain

In [11]:
from langchain_core.load import dumps, loads

In [12]:
def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string for deduplication
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return back to Document objects
    return [loads(doc) for doc in unique_docs]

# Retrieve
question = "What is task decomposition for LLM agents?"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question":question})

print("The retrieved docs are : \n", docs)
print("\nNumber of docs retrieved are : ", len(docs))

The retrieved docs are : 
 [Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', '_id': 'bb0f829133de422c9544bb52fc5c987c', '_collection_name': 'my_documents'}, page_content='Illustration of how HuggingGPT works. (Image source: Shen et al. 2023)\n\nThe system comprises of 4 stages:\n(1) Task planning: LLM works as the brain and parses the user requests into multiple tasks. There are four attributes associated with each task: task type, ID, dependencies, and arguments. They use few-shot examples to guide LLM to do task parsing and planning.\nInstruction:\n\nThe AI assistant can parse user input to several tasks: [{"task": task, "id", task_id, "dep": dependency_task_ids, "args": {"text": text, "image": URL, "audio": URL, "video": URL}}]. The "dep" field denotes the id of the previous task which generates a new resource that the current task relies on. A special tag "-task_id" refers to the generated text image, audio and video in the dependency task with i

C:\Users\ashut\AppData\Local\Temp\ipykernel_8816\720274625.py:8: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]


### Final Generation

In [13]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

In [14]:
# Standard RAG Answer Prompt. So it seems in the final chain, we have to give all the retrieved text and the original
# question to the LLM

template = """
Answer the following question based on this context: {context} 
Question: {question}
""" 

prompt = ChatPromptTemplate.from_template(template)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0) 

# Final Chain
final_rag_chain = (
    {"context": retrieval_chain, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
) 

# Test 
question = "What is task decomposition for LLM agents?"
response = final_rag_chain.invoke({"question":question})

print(response)

Task decomposition for LLM (large language model) agents refers to the process of breaking down complex tasks into smaller, manageable subgoals or steps. This approach enables the agent to handle intricate tasks more efficiently by simplifying them into a series of simpler tasks that can be addressed sequentially or in parallel.

There are several techniques for task decomposition:

1. **Chain of Thought (CoT)**: This prompting technique encourages the model to "think step by step," allowing it to utilize more computational resources at test time to break down hard tasks into smaller, more manageable steps.

2. **Tree of Thoughts**: This method extends CoT by exploring multiple reasoning possibilities at each step. It decomposes the problem into various thought steps and generates multiple thoughts for each step, creating a tree structure. The search process can be conducted using breadth-first search (BFS) or depth-first search (DFS), with each state evaluated by a classifier or throu

## 2) RAG Fusion 

![RAG-Fusion Image](Images/RAG-Fusion.png)

Docs:

* https://github.com/langchain-ai/langchain/blob/master/cookbook/rag_fusion.ipynb?ref=blog.langchain.dev

Blog / repo: 

* https://towardsdatascience.com/forget-rag-the-future-is-rag-fusion-1147298d8ad1

### RAG Fusion generates multiple query variations, retrieves documents for each, and combines results using reciprocal rank fusion to improve retrieval relevance

In [15]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.load import dumps, loads
from langchain_core.runnables import RunnablePassthrough

In [16]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

### RAG-Fusion: Query Generation

In [17]:
template = """You are a helpful assistant that generates multiple search queries based on a single input query.
Generate multiple search queries related to: {question}
Output (4 queries):"""

prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [18]:
# Generates the queries and splits them into a Python list

generate_queries = (
    prompt_rag_fusion 
    | llm
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

In [19]:
# To see the actual multiple queries returned by the LLM 

question = "What is task decomposition for LLM agents?"
LLM_Multiple = generate_queries.invoke({'question' : question})
print(LLM_Multiple)
print("\nNumber or questions given: ", len(LLM_Multiple))

['1. How does task decomposition work in large language model (LLM) agents?', '2. Benefits of task decomposition for LLM agents in AI applications.', '3. Examples of task decomposition techniques used in LLM agents.', '4. Comparing task decomposition strategies for LLM agents and traditional AI systems.']

Number or questions given:  4


### Reciprocal RAG Fusion (RRF) Algorithm

In [20]:
def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents 
        and an optional parameter k used in the RRF formula """
    
    fused_scores = {}

    for docs in results:
        for rank, doc in enumerate(docs):
            
            doc_str = dumps(doc)
        
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            
            fused_scores[doc_str] += 1 / (rank + k)

    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]
    
    return [doc for doc, score in reranked_results]

In [21]:
# Helper function to extract text from Document objects

def format_docs(docs):
    """ Extracts just the page content from the Document objects """
    return "\n\n".join(doc.page_content for doc in docs)

### Assemble Retrieval Chain

In [22]:
# Generate queries -> Map retriever -> Rank via RRF -> Format to clean text

retrieval_chain_rag_fusion = (
    generate_queries 
    | retriever.map() 
    | reciprocal_rank_fusion 
    | format_docs
)

### Final RAG Chain

In [23]:
template = """Answer the following question based on this context: {context} 
Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

In [24]:
# Test the pipeline

question = "What is task decomposition for LLM agents?"
response = final_rag_chain.invoke({"question": question})

print(response)

Task decomposition for LLM (Large Language Model) agents refers to the process of breaking down complex tasks into smaller, more manageable steps or subgoals. This approach allows the agent to plan and execute tasks more effectively by simplifying the overall problem. There are several methods for task decomposition:

1. **Prompting Techniques**: Simple prompts can be used to guide the LLM in identifying the steps needed to complete a task. For example, prompts like "Steps for XYZ.\n1." or "What are the subgoals for achieving XYZ?" can help the model outline the necessary actions.

2. **Task-Specific Instructions**: Providing specific instructions tailored to the task can also facilitate decomposition. For instance, asking the model to "Write a story outline." for writing a novel directs it to focus on the relevant components of that task.

3. **Human Inputs**: Human guidance can be utilized to assist in breaking down tasks, leveraging expert knowledge to ensure that the decomposition 

## 3) Decomposition

### Decomposition breaks a complex question into multiple sub-questions, retrieves answers for each, and then synthesizes them to answer the original question

In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter
from langchain_classic import hub

In [26]:
# Initialize LLM

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [27]:
# Document formatter 

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

### Decomposition Generation

In [28]:
template = """You are a helpful assistant that generates multiple sub-questions related to an input question.
The goal is to break down the input into a set of sub-problems / sub-questions that can be answered in isolation.
Generate multiple search queries related to: {question}
Output (3 queries):"""

prompt_decomposition = ChatPromptTemplate.from_template(template) 

generate_queries_decomposition = ( 
    prompt_decomposition 
    | llm 
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

In [29]:
# Let's test and see how the LLM makes decomposed questions 

question = "What are the main components of an LLM-powered autonomous agent system?"

sub_questions = generate_queries_decomposition.invoke({"question" : question})
print(sub_questions)
print("\nThe number of sub questions made are: ", len(sub_questions))

['1. What are the key architectural components of an LLM-powered autonomous agent system?', '2. How do natural language processing and machine learning integrate within an LLM-powered autonomous agent system?', '3. What role do data input and output mechanisms play in the functionality of an LLM-powered autonomous agent system?']

The number of sub questions made are:  3


### Answer Recursively (Multi-hop Reasoning) (Chain-of-Thought Retrieval) 

![Decomposition_Recursively](Images/Decomposition_Recursively.png)

In [30]:
template_recursive = """Here is the question you need to answer:
---
{question}
---

Here is any available background question + answer pairs:
---
{q_a_pairs}
---

Here is additional context relevant to the question:
---
{context}
---

Use the above context and any background question + answer pairs to answer the question: {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template_recursive)

In [31]:
def format_qa_pair(q, a):
    return f"Question: {q}\nAnswer: {a}"

q_a_pairs = ""

for q in sub_questions:
    rag_chain = (
        {"context": itemgetter("question") | retriever | format_docs, 
         "question": itemgetter("question"),
         "q_a_pairs": itemgetter("q_a_pairs")} 
        | decomposition_prompt
        | llm
        | StrOutputParser()
    )

    answer = rag_chain.invoke({"question": q, "q_a_pairs": q_a_pairs})
    q_a_pair = format_qa_pair(q, answer)
    q_a_pairs = q_a_pairs + "\n---\n" + q_a_pair

print("Recursive Result:\n", q_a_pairs, "\n" + "="*50)

Recursive Result:
 
---
Question: 1. What are the key architectural components of an LLM-powered autonomous agent system?
Answer: The key architectural components of an LLM-powered autonomous agent system include:

1. **LLM as the Core Controller**: The large language model (LLM) serves as the brain of the autonomous agent, enabling it to process natural language, generate responses, and solve problems.

2. **Planning**:
   - **Subgoal and Decomposition**: The agent breaks down complex tasks into smaller, manageable subgoals, facilitating efficient task handling.
   - **Reflection and Refinement**: The agent engages in self-criticism and reflection on past actions, allowing it to learn from mistakes and improve future performance.

3. **Memory**:
   - **Short-term Memory**: Utilizes in-context learning to manage immediate tasks and information.
   - **Long-term Memory**: Enables the agent to retain and recall information over extended periods, often through external vector stores for e

### Answer Individually (Sub-query Generation)

![Decomposition_Individually](Images/Decomposition_Individually.png)

In [32]:
prompt_rag = hub.pull("rlm/rag-prompt")

def retrieve_and_rag(main_question, prompt_rag, sub_question_generator_chain):
    """RAG on each sub-question independently"""
    
    sub_questions = sub_question_generator_chain.invoke({"question": main_question})
    rag_results = []
    
    for sub_question in sub_questions:
        retrieved_docs = retriever.invoke(sub_question)
        
        answer = (
            prompt_rag
            | llm
            | StrOutputParser()
        ).invoke({
            "context": format_docs(retrieved_docs),
            "question": sub_question
        })
        
        rag_results.append(answer)
    
    return rag_results, sub_questions

answers, questions_list = retrieve_and_rag(question, prompt_rag, generate_queries_decomposition)

In [33]:
print("Answers to each sub question: \n", answers)
print("\nNumber of answers: ", len(answers)) 

print("\nSub questions: \n", questions_list)
print("\nNumber of sub questions made by LLM: ", len(questions_list)) 

Answers to each sub question: 
 ['The key components of a large language model (LLM) architecture used in autonomous agents include planning, memory, and a natural language interface. Planning involves breaking down tasks into subgoals and refining actions through self-reflection. Memory allows the agent to learn from past experiences, while the natural language interface facilitates communication with external components.', 'In an LLM-powered autonomous agent system, perception and sensor integration involve the LLM acting as the central controller, processing inputs from various sensors and external APIs to gather information. The agent utilizes short-term and long-term memory to retain and recall relevant data, enhancing its decision-making capabilities. Additionally, self-reflection allows the agent to learn from past actions, improving its performance over time.', "Decision-making and planning are crucial for LLM-based autonomous agents as they enable the breakdown of complex task

`zip(questions_list, answers)` walks both lists side by side: The first question with the first answer, then the second with the second, and so on. Each pass adds one pair to `context`, and `\n` is a line break, so each pair is two lines followed by a blank line.

In [34]:
context = ""
for q, a in zip(questions_list, answers):
    context += f"Question: {q}\nAnswer: {a}\n\n"

In [35]:
print(context)

Question: 1. What are the key components of a large language model (LLM) architecture used in autonomous agents?
Answer: The key components of a large language model (LLM) architecture used in autonomous agents include planning, memory, and a natural language interface. Planning involves breaking down tasks into subgoals and refining actions through self-reflection. Memory allows the agent to learn from past experiences, while the natural language interface facilitates communication with external components.

Question: 2. How do perception and sensor integration function within an LLM-powered autonomous agent system?
Answer: In an LLM-powered autonomous agent system, perception and sensor integration involve the LLM acting as the central controller, processing inputs from various sensors and external APIs to gather information. The agent utilizes short-term and long-term memory to retain and recall relevant data, enhancing its decision-making capabilities. Additionally, self-reflection

In [36]:
# Final Synthesis Prompt

template_synthesis = """Here is a set of Q+A pairs: {context}. 
Use these to synthesize an answer to the following overarching question: {question}
"""
prompt_synthesis = ChatPromptTemplate.from_template(template_synthesis)

final_rag_chain = (
    prompt_synthesis
    | llm
    | StrOutputParser()
)

final_synthesis_answer = final_rag_chain.invoke({"context": context, "question": question})

print("Synthesized Result:\n", final_synthesis_answer)

Synthesized Result:
 An LLM-powered autonomous agent system comprises several main components that work together to enhance its functionality and performance. These components include:

1. **Planning**: This involves breaking down complex tasks into manageable subgoals, allowing the agent to handle tasks efficiently. The integration of external classical planners facilitates long-horizon planning, enabling the generation of structured plans.

2. **Memory**: The system utilizes both short-term and long-term memory to retain and recall relevant information from past experiences. This capability enhances the agent's decision-making and allows it to learn from previous actions.

3. **Natural Language Interface**: This component enables effective communication between the agent and external components, facilitating interaction and information exchange.

4. **Perception and Sensor Integration**: The LLM acts as the central controller, processing inputs from various sensors and external APIs 

## Step-Back

![StepBack Image](Images/StepBack.png)

### Step-Back refers to paraphrasing a specific question into a broader, more generic version to improve retrieval and answer quality in RAG systems

In [37]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [38]:
# Helper function to extract text from Document objects

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

### Few-Shot Examples

In [39]:
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "What can the members of The Police do?",
    },
    {
        "input": "Jan Sindel’s was born in what country?",
        "output": "What is Jan Sindel’s personal history?",
    },
]

In [40]:
# Transform these into example messages for the Chat Model

example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

### Step-Back Question Generator

In [41]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:",
        ),
        
        # Inject the few-shot examples
        few_shot_prompt,
        
        # The new question to process
        ("user", "{question}"),
    ]
)

print("This is how the prompt looks like: \n\n", prompt)

This is how the prompt looks like: 

 input_variables=['question'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:'), additional_kwargs={}), FewShotChatMessagePromptTemplate(examples=[{'input': 'Could the members of The Police perform lawful arrests?', 'output': 'What can the members of The Police do?'}, {'input': 'Jan Sindel’s was born in what country?', 'output': 'What is Jan Sindel’s personal history?'}], input_variables=[], input_types={}, partial_variables={}, example_prompt=ChatPromptTemplate(input_variables=['input', 'output'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, t

In [42]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

generate_queries_step_back = prompt | llm | StrOutputParser()

In [43]:
# Let's test this and see what the generated Step-Back query looks like 

question = "What is task decomposition for LLM agents?"

step_back_query = generate_queries_step_back.invoke({"question": question})
print("The Step-Back query is: \n", step_back_query)

The Step-Back query is: 
 What does task decomposition mean in the context of agents?


### Final Answer Generation Prompt

In [44]:
response_prompt_template = """You are an expert of world knowledge. I am going to ask you a question. 
Your response should be comprehensive and not contradicted with the following context if they are relevant. Otherwise, ignore them if they are not relevant. 

Normal Context: {normal_context}. 

Step-Back Context: {step_back_context}. 

Original Question: {question}. 

Answer: 

"""

response_prompt = ChatPromptTemplate.from_template(response_prompt_template)

### Assemble the Full RAG Chain

In [45]:
chain = (
    {
        # Route 1: Retrieve context using the original, highly-specific question
        "normal_context": itemgetter("question") | retriever | format_docs,
        
        # Route 2: Generate a broad question, then retrieve context using that broad question
        "step_back_context": generate_queries_step_back | retriever | format_docs,
        
        # Route 3: Pass the original question straight through
        "question": itemgetter("question"),
    }
    | response_prompt
    | llm
    | StrOutputParser()
)

### Execution 

In [46]:
question = "What is task decomposition for LLM agents?"

# Execute the final chain
response = chain.invoke({"question": question})
print(response)

Task decomposition for LLM (large language model) agents refers to the process of breaking down complex tasks into smaller, more manageable subgoals or steps. This approach enhances the efficiency and effectiveness of the agent in handling complicated tasks by allowing it to focus on simpler components one at a time. 

There are several techniques and methodologies for task decomposition in LLM agents:

1. **Chain of Thought (CoT)**: This technique encourages the model to think step by step, transforming a large task into a series of smaller, manageable tasks. By prompting the model to articulate its reasoning process, CoT not only aids in task decomposition but also provides insights into the model's thought process.

2. **Tree of Thoughts**: An extension of CoT, this method involves exploring multiple reasoning possibilities at each step. The problem is decomposed into various thought steps, with multiple thoughts generated for each step, creating a tree structure. This allows for a 

## Hypothetical Document Embedding (HyDE)

![HyDE Image](Images/HyDE.png)

### HyDE generates a hypothetical answer to a question using an LLM, embeds it, and uses this embedding to retrieve relevant documents from a vector store

In [47]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

In [48]:
# Extracts the text from the retrieved Document objects

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

### HyDE Document Generation

In [49]:
template_hyde = """Please write a scientific paper passage to answer the question, 
Question: {question}. 

Passage: 
"""

prompt_hyde = ChatPromptTemplate.from_template(template_hyde)

In [50]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Chain that generates the hypothetical document
generate_docs_for_retrieval = (
    prompt_hyde | llm | StrOutputParser() 
)

### Retrieval Pipeline

In [51]:
# This chain generates the fake doc, uses it to search the vector store, and formats the results

retrieval_chain = generate_docs_for_retrieval | retriever | format_docs

### Final RAG Answer Generation

In [52]:
template_rag = """Answer the following question based on this context: {context}. 
Question: {question}.
"""

prompt_rag = ChatPromptTemplate.from_template(template_rag)

In [53]:
# Let's see how the LLM generates a fake hypothetical answer 

question = "What is task decomposition for LLM agents?"

hypothetical_answer = generate_docs_for_retrieval.invoke({"question": question})
print("The hypothetical question is: \n", hypothetical_answer)

The hypothetical question is: 
 **Task Decomposition for Large Language Model Agents**

Task decomposition refers to the systematic breakdown of complex tasks into smaller, more manageable sub-tasks, enabling Large Language Model (LLM) agents to process and execute them more efficiently. This approach is particularly relevant in the context of LLMs, which are designed to understand and generate human-like text based on input prompts. By decomposing tasks, LLM agents can leverage their capabilities more effectively, enhancing both their performance and the quality of their outputs.

In the realm of artificial intelligence, task decomposition serves several critical functions. First, it allows for the identification of distinct components within a larger task, facilitating a clearer understanding of the requirements and objectives. For instance, a complex task such as "writing a research paper" can be decomposed into several sub-tasks, including "conducting literature review," "formulati

### Assemble the Master Chain

In [54]:
final_rag_chain = (
    {
        # Generate fake doc -> Retrieve real docs -> Format to text
        "context": retrieval_chain, 
        
        # Pass the original question through
        "question": itemgetter("question")
    }
    | prompt_rag
    | llm
    | StrOutputParser()
)

### Execution

In [55]:
# Execute the full pipeline

question = "What is task decomposition for LLM agents?"

response = final_rag_chain.invoke({"question": question})
print(response)

Task decomposition for LLM (large language model) agents refers to the process of breaking down complex tasks into smaller, manageable subgoals. This approach enables the agent to handle intricate tasks more efficiently by transforming them into simpler steps. Task decomposition can be achieved through various methods, including:

1. **Simple Prompting**: Using straightforward prompts like "Steps for XYZ.\n1." or "What are the subgoals for achieving XYZ?" to guide the LLM in identifying the necessary steps.

2. **Task-Specific Instructions**: Providing specific instructions tailored to the task at hand, such as "Write a story outline." for writing a novel, which helps the LLM focus on relevant subgoals.

3. **Human Inputs**: Involving human guidance to outline the steps or subgoals required for the task.

By employing task decomposition, LLM agents can systematically approach complex problems, enhancing their problem-solving capabilities and improving the overall quality of their outpu